In [ ]:
pip install transformers torch datasets scikit-learn pandas numpy streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: starlette
    Found existing installation: starlette 1.6.0
    Uninstalling starlette-1.6.0:
      Successfully uninstalled starlette-1.6.0


In [ ]:
from datasets import load_dataset

print("Loading dataset...")

dataset = load_dataset(
    "gouwsxander/wikipedia-human-ai"
)

print("\nDataset loaded!")
print(dataset)

for split in dataset:
    print("\n==============================")
    print("Split:", split)
    print("Columns:", dataset[split].column_names)
    print("First row:")
    print(dataset[split][0])

Loading dataset...

Dataset loaded!
DatasetDict({
    train: Dataset({
        features: ['page_title', 'human_text', 'ai_text'],
        num_rows: 9970
    })
})

Split: train
Columns: ['page_title', 'human_text', 'ai_text']
First row:
{'page_title': 'Donald Trump', 'human_text': "Trump considered a show business career but instead, to be closer to home, enrolled at Fordham University in 1964. He participated in the Reserve Officers' Training Corps program during his first year, attending classes in a military uniform every Wednesday, but dropped it in his second year. He dropped football after three or four weeks and was a mediocre squash and tennis player. His Fordham friends introduced him to golf. His junior year, he transferred to the Wharton School of the University of Pennsylvania, most often commuting to his father's office on weekends, and graduating in May 1968 with a Bachelor of Science in economics. In college he was not the top student he sometimes claimed to be. By the t

# Training the model

In [1]:
import os
import numpy as np
import pandas as pd

from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATASET_NAME = "gouwsxander/wikipedia-human-ai"

MODEL_NAME = "distilbert-base-uncased"

OUTPUT_DIR = "./ai_slop_model"

MAX_LENGTH = 256

RANDOM_STATE = 42


# ============================================================
# 2. LOAD DATASET
# ============================================================

print("\n==========================================")
print("LOADING WIKIPEDIA HUMAN-AI DATASET")
print("==========================================")

dataset = load_dataset(DATASET_NAME)

print("\nDataset:")
print(dataset)


# ============================================================
# 3. GET TRAINING DATA
# ============================================================

df = dataset["train"].to_pandas()

print("\nOriginal dataset:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "page_title",
    "human_text",
    "ai_text"
]


for column in required_columns:

    if column not in df.columns:

        raise ValueError(
            f"Missing required column: {column}"
        )


print(
    "\n✓ All required columns found."
)


# ============================================================
# 5. CREATE HUMAN DATA
# ============================================================

human_df = pd.DataFrame({

    "text": df["human_text"],

    "label": 0

})


print(
    "\nHuman examples:",
    len(human_df)
)


# ============================================================
# 6. CREATE AI DATA
# ============================================================

ai_df = pd.DataFrame({

    "text": df["ai_text"],

    "label": 1

})


print(
    "AI examples:",
    len(ai_df)
)


# ============================================================
# 7. COMBINE HUMAN + AI
# ============================================================

processed_df = pd.concat(

    [
        human_df,
        ai_df
    ],

    ignore_index=True
)


# ============================================================
# 8. REMOVE EMPTY TEXT
# ============================================================

processed_df = processed_df.dropna(

    subset=["text"]
)


processed_df["text"] = (

    processed_df["text"]
    .astype(str)
    .str.strip()

)


processed_df = processed_df[
    processed_df["text"] != ""
]


# ============================================================
# 9. SHUFFLE DATA
# ============================================================

processed_df = processed_df.sample(

    frac=1,

    random_state=RANDOM_STATE

).reset_index(drop=True)


# ============================================================
# 10. DATASET INFORMATION
# ============================================================

print(
    "\n=========================================="
)

print(
    "PROCESSED DATASET"
)

print(
    "=========================================="
)


print(
    "\nTotal examples:",
    len(processed_df)
)


print(
    "\nLabel distribution:"
)


print(
    processed_df["label"].value_counts()
)


print(
    "\nExpected:"
)

print(
    "0 = Human-written"
)

print(
    "1 = AI-generated"
)


# ============================================================
# 11. TRAIN / VALIDATION SPLIT
# ============================================================

train_df, validation_df = train_test_split(

    processed_df,

    test_size=0.20,

    random_state=RANDOM_STATE,

    stratify=processed_df["label"]
)


print(
    "\nTraining examples:",
    len(train_df)
)


print(
    "Validation examples:",
    len(validation_df)
)


# ============================================================
# 12. CONVERT TO HUGGING FACE DATASETS
# ============================================================

train_dataset = Dataset.from_pandas(

    train_df[
        ["text", "label"]
    ],

    preserve_index=False

)


validation_dataset = Dataset.from_pandas(

    validation_df[
        ["text", "label"]
    ],

    preserve_index=False

)


# ============================================================
# 13. LOAD TOKENIZER
# ============================================================

print(
    "\n=========================================="
)

print(
    "LOADING TOKENIZER"
)

print(
    "=========================================="
)


tokenizer = AutoTokenizer.from_pretrained(

    MODEL_NAME
)


# ============================================================
# 14. TOKENIZATION
# ============================================================

def tokenize_function(examples):

    return tokenizer(

        examples["text"],

        padding="max_length",

        truncation=True,

        max_length=MAX_LENGTH

    )


print(
    "\nTokenizing training data..."
)


train_dataset = train_dataset.map(

    tokenize_function,

    batched=True

)


print(
    "Tokenizing validation data..."
)


validation_dataset = validation_dataset.map(

    tokenize_function,

    batched=True

)


print(
    "✓ Tokenization complete."
)


# ============================================================
# 15. LOAD DISTILBERT
# ============================================================

print(
    "\n=========================================="
)

print(
    "LOADING DISTILBERT"
)

print(
    "=========================================="
)


model = AutoModelForSequenceClassification.from_pretrained(

    MODEL_NAME,

    num_labels=2

)


# ============================================================
# 16. METRICS
# ============================================================

def compute_metrics(pred):

    predictions = np.argmax(

        pred.predictions,

        axis=1

    )

    labels = pred.label_ids


    precision, recall, f1, _ = (

        precision_recall_fscore_support(

            labels,

            predictions,

            average="binary",

            zero_division=0

        )

    )


    accuracy = accuracy_score(

        labels,

        predictions

    )


    return {

        "accuracy": accuracy,

        "precision": precision,

        "recall": recall,

        "f1": f1

    }


# ============================================================
# 17. TRAINING SETTINGS
# ============================================================

training_args = TrainingArguments(

    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    logging_steps=100,

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    save_total_limit=2,

    report_to="none"

)


# ============================================================
# 18. CREATE TRAINER
# ============================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=validation_dataset,

    processing_class=tokenizer,

    compute_metrics=compute_metrics

)


# ============================================================
# 19. TRAIN
# ============================================================

print(
    "\n=========================================="
)

print(
    "STARTING TRAINING"
)

print(
    "=========================================="
)


trainer.train()


# ============================================================
# 20. EVALUATE
# ============================================================

print(
    "\n=========================================="
)

print(
    "EVALUATING MODEL"
)

print(
    "=========================================="
)


results = trainer.evaluate()


print("\nEvaluation results:")


for key, value in results.items():

    print(
        f"{key}: {value}"
    )


# ============================================================
# 21. CONFUSION MATRIX
# ============================================================

print(
    "\n=========================================="
)

print(
    "CONFUSION MATRIX"
)

print(
    "=========================================="
)


predictions = trainer.predict(

    validation_dataset

)


predicted_labels = np.argmax(

    predictions.predictions,

    axis=1

)


true_labels = predictions.label_ids


cm = confusion_matrix(

    true_labels,

    predicted_labels

)


print(cm)


# ============================================================
# 22. CLASSIFICATION REPORT
# ============================================================

print(
    "\n=========================================="
)

print(
    "CLASSIFICATION REPORT"
)

print(
    "=========================================="
)


print(

    classification_report(

        true_labels,

        predicted_labels,

        target_names=[

            "Human",

            "AI"

        ],

        zero_division=0

    )

)


# ============================================================
# 23. SAVE MODEL
# ============================================================

print(
    "\n=========================================="
)

print(
    "SAVING MODEL"
)

print(
    "=========================================="
)


trainer.save_model(

    OUTPUT_DIR

)


tokenizer.save_pretrained(

    OUTPUT_DIR

)


# ============================================================
# 24. VERIFY MODEL DIRECTORY
# ============================================================

print(
    "\nChecking saved model..."
)


if os.path.exists(OUTPUT_DIR):

    print(
        "\n✓ MODEL SAVED SUCCESSFULLY!"
    )

    print(
        "\nModel location:"
    )

    print(
        os.path.abspath(
            OUTPUT_DIR
        )
    )

    print(
        "\nFiles:"
    )

    for file in os.listdir(OUTPUT_DIR):

        print(
            "  ✓",
            file
        )

else:

    raise RuntimeError(

        "Model directory was not created."
    )


print(
    "\n=========================================="
)

print(
    "TRAINING COMPLETE"
)

print(
    "=========================================="
)


LOADING WIKIPEDIA HUMAN-AI DATASET


README.md:   0%|          | 0.00/236 [00:00<?, ?B/s]

wikipedia.jsonl: reconstructing file:   0%|          |  0.00B / 10.5MB            

wikipedia.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9970 [00:00<?, ? examples/s]

...

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.024216,0.217287,3,0.950602,0.966269,0.933801,0.949758



Evaluation results:
eval_loss: 0.21728745102882385
eval_accuracy: 0.9506018054162487
eval_precision: 0.9662688116242865
eval_recall: 0.9338014042126379
eval_f1: 0.9497577148686559

CONFUSION MATRIX


[[1929   65]
 [ 132 1862]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

       Human       0.94      0.97      0.95      1994
          AI       0.97      0.93      0.95      1994

    accuracy                           0.95      3988
   macro avg       0.95      0.95      0.95      3988
weighted avg       0.95      0.95      0.95      3988


SAVING MODEL


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Checking saved model...

✓ MODEL SAVED SUCCESSFULLY!

Model location:
/content/ai_slop_model

Files:
  ✓ training_args.bin
  ✓ config.json
  ✓ model.safetensors
  ✓ tokenizer.json
  ✓ checkpoint-5982
  ✓ checkpoint-3988
  ✓ tokenizer_config.json

TRAINING COMPLETE


In [2]:
import os
print(os.path.exists("./ai_slop_model"))

True


# Detector

In [ ]:
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# ============================================================
# 1. MODEL PATH
# ============================================================

MODEL_PATH = os.path.abspath("./ai_slop_model")


# ============================================================
# 2. CHECK MODEL
# ============================================================

print("\n========================================")
print("        AI SLOP DETECTOR")
print("========================================")

print("\nModel path:")
print(MODEL_PATH)


if not os.path.exists(MODEL_PATH):

    raise FileNotFoundError(
        "\nModel not found!\n"
        "Please run train_model.py first."
    )


print("\nModel files:")

for file in os.listdir(MODEL_PATH):

    print("  ✓", file)


# ============================================================
# 3. LOAD TOKENIZER
# ============================================================

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)


# ============================================================
# 4. LOAD TRAINED MODEL
# ============================================================

print("Loading trained model...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model.eval()

print("\n✓ Model loaded successfully!")


# ============================================================
# 5. DETECTION FUNCTION
# ============================================================

def detect_ai(text):

    # Check text length

    if not text or len(text.strip()) < 20:

        return {
            "label": "Insufficient text",
            "ai_probability": 0.0,
            "human_probability": 0.0
        }


    # --------------------------------------------------------
    # TOKENIZE
    # --------------------------------------------------------

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )


    # --------------------------------------------------------
    # MODEL PREDICTION
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = model(**inputs)

        probabilities = torch.softmax(
            outputs.logits,
            dim=1
        )


    # --------------------------------------------------------
    # GET PROBABILITIES
    # --------------------------------------------------------

    human_probability = probabilities[0][0].item()

    ai_probability = probabilities[0][1].item()


    # --------------------------------------------------------
    # CLASSIFICATION
    # --------------------------------------------------------

    if ai_probability >= 0.70:

        label = "AI-generated"

    elif ai_probability <= 0.30:

        label = "Human-written"

    else:

        label = "Mixed / Uncertain"


    return {

        "label": label,

        "ai_probability": round(
            ai_probability * 100,
            2
        ),

        "human_probability": round(
            human_probability * 100,
            2
        )
    }


# ============================================================
# 6. TEST THE DETECTOR
# ============================================================

print("\n========================================")
print("Enter text for analysis")
print("Type 'exit' to stop")
print("========================================")


while True:

    text = input(
        "\nPaste your text:\n> "
    )


    if text.lower() == "exit":

        print("\nDetector stopped.")

        break


    result = detect_ai(text)


    print("\n----------------------------------------")
    print("RESULT")
    print("----------------------------------------")

    print(
        "Classification:",
        result["label"]
    )

    print(
        "AI Probability:",
        f'{result["ai_probability"]}%'
    )

    print(
        "Human Probability:",
        f'{result["human_probability"]}%'
    )

    print("----------------------------------------")


        AI SLOP DETECTOR

Model path:
/content/ai_slop_model

Model files:
  ✓ training_args.bin
  ✓ config.json
  ✓ model.safetensors
  ✓ tokenizer.json
  ✓ checkpoint-5982
  ✓ checkpoint-3988
  ✓ tokenizer_config.json

Loading tokenizer...
Loading trained model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


✓ Model loaded successfully!

Enter text for analysis
Type 'exit' to stop

----------------------------------------
RESULT
----------------------------------------
Classification: Human-written
AI Probability: 0.04%
Human Probability: 99.96%
----------------------------------------

----------------------------------------
RESULT
----------------------------------------
Classification: AI-generated
AI Probability: 96.32%
Human Probability: 3.68%
----------------------------------------


# Building the website

In [1]:
import os
import torch
import streamlit as st

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="AI Slop Detector",
    page_icon="🤖",
    layout="centered"
)


# ============================================================
# MODEL PATH
# ============================================================

MODEL_PATH = os.path.abspath(
    "./ai_slop_model"
)


# ============================================================
# LOAD MODEL
# ============================================================

@st.cache_resource
def load_model():

    if not os.path.exists(MODEL_PATH):

        st.error(
            "Trained model not found. "
            "Run train_model.py first."
        )

        st.stop()


    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        local_files_only=True
    )


    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        local_files_only=True
    )


    model.eval()


    return tokenizer, model


tokenizer, model = load_model()


# ============================================================
# HEADER
# ============================================================

st.title("🤖 AI Slop Detector")

st.write(
    "Analyze text using a transformer-based "
    "AI-generated text classifier."
)

st.warning(
    "The result is a probability estimate, "
    "not definitive proof that AI wrote the text."
)


# ============================================================
# TEXT INPUT
# ============================================================

text = st.text_area(

    "Enter or paste your text:",

    height=300,

    placeholder=(
        "Paste an article, essay, paragraph, "
        "or social media post here..."
    )
)


# ============================================================
# DETECT BUTTON
# ============================================================

if st.button(
    "🔍 Detect AI",
    use_container_width=True
):

    if len(text.strip()) < 20:

        st.error(
            "Please enter at least 20 characters."
        )

    else:

        with st.spinner(
            "Analyzing text..."
        ):

            # ----------------------------------------------
            # TOKENIZE
            # ----------------------------------------------

            inputs = tokenizer(

                text,

                return_tensors="pt",

                truncation=True,

                max_length=512,

                padding=True
            )


            # ----------------------------------------------
            # PREDICTION
            # ----------------------------------------------

            with torch.no_grad():

                outputs = model(
                    **inputs
                )


                probabilities = torch.softmax(
                    outputs.logits,
                    dim=1
                )


            human_score = probabilities[0][0].item()

            ai_score = probabilities[0][1].item()


        # ====================================================
        # CLASSIFICATION
        # ====================================================

        if ai_score >= 0.70:

            classification = "🤖 AI-generated"

        elif ai_score <= 0.30:

            classification = "👤 Human-written"

        else:

            classification = "⚠️ Mixed / Uncertain"


        # ====================================================
        # RESULTS
        # ====================================================

        st.divider()

        st.subheader(
            classification
        )


        # ====================================================
        # PROBABILITY METRICS
        # ====================================================

        col1, col2 = st.columns(2)


        with col1:

            st.metric(
                "AI Probability",
                f"{ai_score * 100:.2f}%"
            )


        with col2:

            st.metric(
                "Human Probability",
                f"{human_score * 100:.2f}%"
            )


        # ====================================================
        # AI PROBABILITY BAR
        # ====================================================

        st.write(
            "AI-generated probability"
        )

        st.progress(
            ai_score
        )


        # ====================================================
        # INTERPRETATION
        # ====================================================

        st.divider()

        st.subheader(
            "Interpretation"
        )


        if ai_score >= 0.70:

            st.info(
                "The model considers this text "
                "more similar to the AI-generated "
                "examples it learned during training."
            )

        elif ai_score <= 0.30:

            st.info(
                "The model considers this text "
                "more similar to the human-written "
                "examples it learned during training."
            )

        else:

            st.info(
                "The model is uncertain. The text "
                "falls between the two classification "
                "thresholds."
            )


        # ====================================================
        # TEXT STATISTICS
        # ====================================================

        st.divider()

        st.subheader(
            "Text Statistics"
        )


        words = len(
            text.split()
        )

        characters = len(
            text
        )

        sentences = max(
            len(text.split(".")) - 1,
            1
        )


        col1, col2, col3 = st.columns(3)


        with col1:

            st.metric(
                "Words",
                words
            )


        with col2:

            st.metric(
                "Sentences",
                sentences
            )


        with col3:

            st.metric(
                "Characters",
                characters
            )

ModuleNotFoundError: No module named 'streamlit'